# Task 1.2 — Key Assumptions (8 marks)

Identification of at least three assumptions that the PLTM’s *contribution, architecture, or algorithm* depend on (not general ML assumptions like i.i.d. data).



## Key assumptions (PLTM)

In the Pouch Latent Tree Model (PLTM), the following three assumptions are built into the *architecture* and *algorithm* of the paper. They are not generic ML assumptions (like “data is i.i.d.”) but specific to this method.

---

### Assumption 1

**Assumption:**  
Each observed (manifest) variable can have **only one parent** in the model—exactly one discrete latent variable. So every attribute sits in exactly one “pouch” and is tied to one latent.

**Why the method needs it:**  
The whole idea of a “pouch” is a *group* of manifest variables under one latent. If one attribute could sit under two latents, the tree-and-pouch picture would break. The structure-search algorithm also relies on this when it uses “pouching” (merge two pouches) and “unpouching” (split a pouch); those moves only make sense when each variable belongs to one pouch.

**Violation scenario:**  
Imagine one observed variable that is really driven by *two* hidden factors at once—e.g. “monthly spending” depends on both “income level” and “frugality.” The method cannot represent that directly, because it would require that variable to be under two latents.

**Paper reference:** Section 2 (definition of pouch nodes and tree; each manifest in one pouch). Figure 1(a) shows an example where each X_i belongs to a single pouch.

---

### Assumption 2

**Assumption:**  
The latent variables and pouches form a **rooted tree**: every node has at most one parent, and there are **no cycles**. So the dependency structure is a strict hierarchy, not a general graph.

**Why the method needs it:**  
The E-step of EM needs to compute probabilities over the latent variables given the data. The paper does this with *exact* inference for mixed (discrete + Gaussian) Bayesian networks. That algorithm (Lauritzen & Jensen style) is efficient only when the graph is a tree (or similar). If there were cycles or multiple parents, we would need a different, heavier inference method and the current algorithm would not apply as stated.

**Violation scenario:**  
Think of a system where hidden factors influence each other in a loop or a dense network—e.g. in biology or economics, where “latent state A” affects “latent state B” and B also affects A, or many latents interact in a web. That cannot be represented as a rooted tree.

**Paper reference:** Section 2 (PLTM defined as a rooted tree). Section 3 (E-step uses exact inference for the mixed network, which assumes the tree structure).

---

### Assumption 3

**Assumption:**  
Inside each pouch, the **continuous** manifest variables follow a **Gaussian (normal) distribution** given the value of their parent latent. So for each latent state *y*, the observations in that pouch are modeled as P(x|y) = N(x | μ_y, Σ_y).

**Why the method needs it:**  
The EM updates in the paper are the usual ones for Gaussian mixtures: in the M-step, means and covariances have closed-form formulas from the posterior counts. The paper also adds an *eigenvalue constraint* on the covariance matrix to avoid degenerate solutions; that constraint is defined for Gaussian covariances. If the data given the latent were not Gaussian (e.g. counts or heavy-tailed), these update equations and the constraint would not be the right model and the algorithm would not be justified as written.

**Violation scenario:**  
Consider data that are counts (e.g. number of clicks, purchases per day) or strongly skewed (e.g. power-law or Poisson-like). In their natural scale, such variables are not Gaussian. If we still use PLTM as-is, the Gaussian assumption is violated and the model may fit poorly or the “facets” may be misleading.

**Paper reference:** Section 2 (“We assume that, given a value y of Y, X follow the conditional Gaussian distribution P(x|y) = N(x|μ_y, Σ_y)”). Section 3 (M-step updates for μ_y and Σ_y, and the eigenvalue constraint on Σ_y).